In [2]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load match summaries and ball-by-ball delivery datasets
matches = pd.read_csv(r"c:\Users\hp\Downloads\archive (9)\matches.csv")
deliveries = pd.read_csv(r"c:\Users\hp\Downloads\archive (9)\deliveries.csv")



In [3]:
# Standardize historical franchise names to current team names
old_to_new = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
}

matches = matches.replace(old_to_new)
deliveries = deliveries.replace(old_to_new)

# Filter for the 10 currently active IPL teams
teams = [
    'Sunrisers Hyderabad', 'Mumbai Indians', 'Royal Challengers Bengaluru',
    'Kolkata Knight Riders', 'Punjab Kings', 'Chennai Super Kings',
    'Rajasthan Royals', 'Delhi Capitals', 'Gujarat Titans', 'Lucknow Super Giants'
]

matches = matches[matches['team1'].isin(teams) & matches['team2'].isin(teams)]
matches = matches.dropna(subset=['winner', 'target_runs'])
matches['city'] = matches['city'].fillna('UAE').replace({'Bangalore': 'Bengaluru'})

# Merge ball-by-ball deliveries with match-level target and outcome data
df = deliveries.merge(matches[['id', 'city', 'winner', 'target_runs']], left_on='match_id', right_on='id')

# Filter exclusively for second-innings chasing scenarios
df = df[df['inning'] == 2].copy()
df = df[df['batting_team'].isin(teams) & df['bowling_team'].isin(teams)]


In [4]:
# 1. Calculate cumulative runs scored per match
df['current_score'] = df.groupby('match_id')['total_runs'].cumsum()

# 2. Calculate remaining runs required to win
df['runs_left'] = df['target_runs'] - df['current_score']

# 3. Calculate remaining legal deliveries (out of 120 balls)
df['balls_left'] = 120 - (df['over'] * 6 + df['ball'])

# 4. Calculate wickets remaining in hand (out of 10 wickets)
wickets_fallen = df.groupby('match_id')['is_wicket'].cumsum()
df['wickets_left'] = 10 - wickets_fallen

# 5. Calculate Current Run Rate (CRR) and Required Run Rate (RRR)
balls_bowled = 120 - df['balls_left']
df['crr'] = np.where(balls_bowled > 0, (df['current_score'] * 6) / balls_bowled, 0)
df['rrr'] = np.where(df['balls_left'] > 0, (df['runs_left'] * 6) / df['balls_left'], 0)

# 6. Create binary target column (1 if chasing team won, 0 otherwise)
df['result'] = (df['batting_team'] == df['winner']).astype(int)


In [5]:
# 1. Select relevant features and remove invalid or missing records
final_df = df[['batting_team', 'bowling_team', 'city', 'runs_left', 'balls_left', 'wickets_left', 'target_runs', 'crr', 'rrr', 'result']]
final_df = final_df[final_df['balls_left'] >= 0].dropna()

# 2. Convert categorical text columns into binary dummy variables (One-Hot Encoding)
final_df_encoded = pd.get_dummies(final_df, columns=['batting_team', 'bowling_team', 'city'], drop_first=True)

# 3. Separate input features (X) and target variable (y)
X = final_df_encoded.drop(columns=['result'])
y = final_df_encoded['result']

# 4. Split data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Scale numerical features for optimal model convergence
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Train the Logistic Regression classifier
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# 7. Evaluate model accuracy on test data
y_pred = model.predict(X_test_scaled)
print("Model Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")

# 8. Export trained model, scaler, and feature columns for deployment
pickle.dump(model, open('ipl_model.pkl', 'wb'))
pickle.dump(scaler, open('ipl_scaler.pkl', 'wb'))
pickle.dump(X.columns.tolist(), open('ipl_columns.pkl', 'wb'))
print("All files saved successfully!")


Model Accuracy: 79.8 %
All files saved successfully!
